# UTKFace Age-Gender Baseline\n\nNotebook này gom toàn bộ baseline pipeline vào 1 file để chạy trên Kaggle.\n\nMục tiêu:\n- Multi-task: `gender classification + age regression`\n- So sánh công bằng 3 backbone: `EfficientNetB0`, `ResNet50`, `MobileNetV3Large`\n- Cùng split, cùng preprocessing pipeline, cùng head, cùng train recipe\n- Hỗ trợ `phase 1` freeze backbone và `phase 2` fine-tuning\n- Metric chính: `gender accuracy`, `gender F1`, `age MAE`\n

In [ ]:
import json\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport tensorflow as tf\nfrom sklearn.model_selection import train_test_split\n\nprint('TensorFlow version:', tf.__version__)\ntf.keras.utils.set_random_seed(42)\n

## 1. Config\n\nChỉ cần chỉnh các đường dẫn và hyperparameter ở cell dưới.\n

In [ ]:
@dataclass\nclass ExperimentConfig:\n    dataset_dir: str\n    output_dir: str = '/kaggle/working/artifacts'\n    image_size: int = 128\n    batch_size: int = 64\n    seed: int = 42\n    val_size: float = 0.15\n    test_size: float = 0.15\n    phase1_epochs: int = 20\n    phase2_epochs: int = 10\n    fine_tune_layers: int = 40\n    head_learning_rate: float = 1e-3\n    fine_tune_learning_rate: float = 5e-5\n    gender_loss_weight: float = 1.0\n    age_loss_weight: float = 0.3\n    early_stopping_patience: int = 6\n    reduce_lr_patience: int = 3\n    min_learning_rate: float = 1e-6\n    train_shuffle_buffer: int = 2048\n    freeze_batch_norm: bool = True\n\n\nDATASET_DIR = '/kaggle/input/utkface-new/UTKFace'\n# Ví dụ khác nếu dataset của bạn có path khác trên Kaggle:\n# DATASET_DIR = '/kaggle/input/datasets/jangedoo/utkface-new/UTKFace'\n\nconfig = ExperimentConfig(dataset_dir=DATASET_DIR)\nBACKBONES = ['efficientnetb0', 'resnet50', 'mobilenetv3large']\n\nPath(config.output_dir).mkdir(parents=True, exist_ok=True)\nprint(config)\n

## 2. Dataset Scan và Split\n\nTạo split một lần, sau đó tái sử dụng cho cả 3 backbone để so sánh công bằng.\n

In [ ]:
def is_valid_utkface_name(path: Path) -> bool:\n    parts = path.stem.split('_')\n    if len(parts) < 4:\n        return False\n    try:\n        int(parts[0])\n        gender = int(parts[1])\n    except ValueError:\n        return False\n    return gender in (0, 1)\n\n\ndef scan_utkface_dataset(dataset_dir: str) -> pd.DataFrame:\n    dataset_path = Path(dataset_dir)\n    image_paths = sorted(p for p in dataset_path.glob('*.jpg') if is_valid_utkface_name(p))\n\n    rows = []\n    for path in image_paths:\n        parts = path.stem.split('_')\n        age = int(parts[0])\n        gender = int(parts[1])\n        age_bin = min(age // 10, 11)\n        rows.append({\n            'filepath': str(path.resolve()),\n            'age': age,\n            'gender': gender,\n            'age_bin': age_bin,\n            'stratify_key': f'{gender}_{age_bin}',\n        })\n\n    if not rows:\n        raise FileNotFoundError(f'No valid UTKFace .jpg files found in {dataset_dir}')\n\n    return pd.DataFrame(rows)\n\n\ndef safe_stratify_labels(frame: pd.DataFrame):\n    counts = frame['stratify_key'].value_counts()\n    if counts.empty or counts.min() < 2:\n        return None\n    return frame['stratify_key']\n\n\ndef split_dataframe(data: pd.DataFrame, val_size: float, test_size: float, seed: int):\n    holdout_size = val_size + test_size\n    train_df, holdout_df = train_test_split(\n        data,\n        test_size=holdout_size,\n        random_state=seed,\n        shuffle=True,\n        stratify=safe_stratify_labels(data),\n    )\n\n    val_df, test_df = train_test_split(\n        holdout_df,\n        test_size=test_size / holdout_size,\n        random_state=seed,\n        shuffle=True,\n        stratify=safe_stratify_labels(holdout_df),\n    )\n\n    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)\n\n\ndef save_splits(split_dir: Path, train_df: pd.DataFrame, val_df: pd.DataFrame, test_df: pd.DataFrame):\n    split_dir.mkdir(parents=True, exist_ok=True)\n    train_df.to_csv(split_dir / 'train.csv', index=False)\n    val_df.to_csv(split_dir / 'val.csv', index=False)\n    test_df.to_csv(split_dir / 'test.csv', index=False)\n\n\ndef load_splits(split_dir: Path):\n    train_path = split_dir / 'train.csv'\n    val_path = split_dir / 'val.csv'\n    test_path = split_dir / 'test.csv'\n    if not all(p.exists() for p in [train_path, val_path, test_path]):\n        return None\n    return pd.read_csv(train_path), pd.read_csv(val_path), pd.read_csv(test_path)\n\n\ndef describe_split(frame: pd.DataFrame) -> dict:\n    return {\n        'samples': int(len(frame)),\n        'male_ratio': float((frame['gender'] == 0).mean()),\n        'female_ratio': float((frame['gender'] == 1).mean()),\n        'age_mean': float(frame['age'].mean()),\n        'age_std': float(frame['age'].std(ddof=0)),\n        'age_min': int(frame['age'].min()),\n        'age_max': int(frame['age'].max()),\n    }\n\n\nsplit_dir = Path(config.output_dir) / 'splits'\nexisting_splits = load_splits(split_dir)\n\nif existing_splits is None:\n    full_df = scan_utkface_dataset(config.dataset_dir)\n    train_df, val_df, test_df = split_dataframe(full_df, config.val_size, config.test_size, config.seed)\n    save_splits(split_dir, train_df, val_df, test_df)\nelse:\n    train_df, val_df, test_df = existing_splits\n\nsplit_summary = {\n    'train': describe_split(train_df),\n    'val': describe_split(val_df),\n    'test': describe_split(test_df),\n}\n\nprint(json.dumps(split_summary, indent=2))\n

## 3. Preprocessing và Augmentation\n\nGiữ cùng data pipeline cho mọi backbone. `preprocess_input` riêng từng backbone sẽ được đặt trong model builder.\n

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE\n\n\ndef build_augmentation():\n    return tf.keras.Sequential([\n        tf.keras.layers.RandomFlip('horizontal'),\n        tf.keras.layers.RandomRotation(0.08),\n        tf.keras.layers.RandomZoom(height_factor=0.1, width_factor=0.1),\n        tf.keras.layers.RandomContrast(0.1),\n    ], name='augmentation')\n\n\ndef decode_image(path, image_size: int):\n    image_bytes = tf.io.read_file(path)\n    image = tf.io.decode_jpeg(image_bytes, channels=3)\n    image = tf.image.resize(image, (image_size, image_size))\n    return tf.cast(image, tf.float32)\n\n\ndef make_dataset(frame: pd.DataFrame, image_size: int, batch_size: int, training: bool, shuffle_buffer: int, seed: int):\n    paths = frame['filepath'].astype(str).to_numpy()\n    genders = frame['gender'].astype('float32').to_numpy()\n    ages = frame['age'].astype('float32').to_numpy()\n\n    ds = tf.data.Dataset.from_tensor_slices((paths, genders, ages))\n    if training:\n        ds = ds.shuffle(min(shuffle_buffer, len(frame)), seed=seed, reshuffle_each_iteration=True)\n\n    def _load(path, gender, age):\n        image = decode_image(path, image_size=image_size)\n        labels = {\n            'gender_output': tf.expand_dims(gender, axis=-1),\n            'age_output': tf.expand_dims(age, axis=-1),\n        }\n        return image, labels\n\n    ds = ds.map(_load, num_parallel_calls=AUTOTUNE)\n\n    if training:\n        aug = build_augmentation()\n        ds = ds.map(lambda image, labels: (aug(image, training=True), labels), num_parallel_calls=AUTOTUNE)\n\n    return ds.batch(batch_size).prefetch(AUTOTUNE)\n\n\ntrain_ds = make_dataset(train_df, config.image_size, config.batch_size, True, config.train_shuffle_buffer, config.seed)\nval_ds = make_dataset(val_df, config.image_size, config.batch_size, False, config.train_shuffle_buffer, config.seed)\ntest_ds = make_dataset(test_df, config.image_size, config.batch_size, False, config.train_shuffle_buffer, config.seed)\n\nfor images, labels in train_ds.take(1):\n    print('images:', images.shape, images.dtype)\n    print('gender_output:', labels['gender_output'].shape)\n    print('age_output:', labels['age_output'].shape)\n

## 4. Model Building\n\nBackbone thay đổi qua registry, còn head multi-task được giữ nguyên để đảm bảo fair comparison.\n

In [ ]:
class BinaryF1Score(tf.keras.metrics.Metric):\n    def __init__(self, name='f1', threshold=0.5, **kwargs):\n        super().__init__(name=name, **kwargs)\n        self.threshold = threshold\n        self.true_positives = self.add_weight(name='tp', initializer='zeros')\n        self.false_positives = self.add_weight(name='fp', initializer='zeros')\n        self.false_negatives = self.add_weight(name='fn', initializer='zeros')\n\n    def update_state(self, y_true, y_pred, sample_weight=None):\n        y_true = tf.cast(y_true, tf.float32)\n        y_pred = tf.cast(y_pred >= self.threshold, tf.float32)\n        y_true = tf.reshape(y_true, [-1])\n        y_pred = tf.reshape(y_pred, [-1])\n\n        tp = tf.reduce_sum(y_true * y_pred)\n        fp = tf.reduce_sum((1.0 - y_true) * y_pred)\n        fn = tf.reduce_sum(y_true * (1.0 - y_pred))\n\n        self.true_positives.assign_add(tp)\n        self.false_positives.assign_add(fp)\n        self.false_negatives.assign_add(fn)\n\n    def result(self):\n        numerator = 2.0 * self.true_positives\n        denominator = numerator + self.false_positives + self.false_negatives\n        return tf.math.divide_no_nan(numerator, denominator)\n\n    def reset_state(self):\n        for variable in self.variables:\n            variable.assign(0.0)\n\n\nAVAILABLE_BACKBONES = {\n    'efficientnetb0': {\n        'name': 'EfficientNetB0',\n        'builder': tf.keras.applications.EfficientNetB0,\n        'preprocess_input': tf.keras.applications.efficientnet.preprocess_input,\n    },\n    'resnet50': {\n        'name': 'ResNet50',\n        'builder': tf.keras.applications.ResNet50,\n        'preprocess_input': tf.keras.applications.resnet50.preprocess_input,\n    },\n    'mobilenetv3large': {\n        'name': 'MobileNetV3Large',\n        'builder': tf.keras.applications.MobileNetV3Large,\n        'preprocess_input': tf.keras.applications.mobilenet_v3.preprocess_input,\n    },\n}\n\n\ndef build_multitask_model(backbone_name: str, image_size: int, weights='imagenet', pooling='avg'):\n    backbone_info = AVAILABLE_BACKBONES[backbone_name]\n    inputs = tf.keras.layers.Input(shape=(image_size, image_size, 3), name='image')\n\n    x = backbone_info['preprocess_input'](inputs)\n    backbone = backbone_info['builder'](\n        include_top=False,\n        weights=weights,\n        pooling=pooling,\n        input_shape=(image_size, image_size, 3),\n    )\n    backbone.trainable = False\n    x = backbone(x, training=False)\n\n    shared = tf.keras.layers.BatchNormalization(name='shared_bn')(x)\n    shared = tf.keras.layers.Dense(256, activation='relu', name='shared_dense')(shared)\n    shared = tf.keras.layers.Dropout(0.3, name='shared_dropout')(shared)\n\n    gender = tf.keras.layers.Dense(128, activation='relu', name='gender_dense')(shared)\n    gender = tf.keras.layers.Dropout(0.2, name='gender_dropout')(gender)\n    gender_output = tf.keras.layers.Dense(1, activation='sigmoid', name='gender_output')(gender)\n\n    age = tf.keras.layers.Dense(128, activation='relu', name='age_dense')(shared)\n    age = tf.keras.layers.Dropout(0.2, name='age_dropout')(age)\n    age_output = tf.keras.layers.Dense(1, activation='linear', name='age_output')(age)\n\n    model = tf.keras.Model(\n        inputs=inputs,\n        outputs={'gender_output': gender_output, 'age_output': age_output},\n        name=f"{backbone_info['name']}_AgeGender",\n    )\n    return model, backbone\n\n\ndef compile_model(model, learning_rate: float, gender_loss_weight: float, age_loss_weight: float):\n    model.compile(\n        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),\n        loss={\n            'gender_output': tf.keras.losses.BinaryCrossentropy(),\n            'age_output': tf.keras.losses.Huber(),\n        },\n        loss_weights={\n            'gender_output': gender_loss_weight,\n            'age_output': age_loss_weight,\n        },\n        metrics={\n            'gender_output': [\n                tf.keras.metrics.BinaryAccuracy(name='accuracy'),\n                BinaryF1Score(name='f1'),\n            ],\n            'age_output': [tf.keras.metrics.MeanAbsoluteError(name='mae')],\n        },\n    )\n\n\ndef configure_fine_tuning(backbone, fine_tune_layers: int, freeze_batch_norm: bool = True):\n    backbone.trainable = True\n\n    if fine_tune_layers > 0:\n        for layer in backbone.layers[:-fine_tune_layers]:\n            layer.trainable = False\n\n    for layer in backbone.layers[-fine_tune_layers:] if fine_tune_layers > 0 else backbone.layers:\n        if freeze_batch_norm and isinstance(layer, tf.keras.layers.BatchNormalization):\n            layer.trainable = False\n        else:\n            layer.trainable = True\n

## 5. Evaluation Helpers\n

In [ ]:
def flatten_batches(dataset):\n    genders, ages = [], []\n    for _, labels in dataset:\n        genders.append(labels['gender_output'].numpy().reshape(-1))\n        ages.append(labels['age_output'].numpy().reshape(-1))\n    return np.concatenate(genders), np.concatenate(ages)\n\n\ndef binary_f1_score_np(y_true, y_pred):\n    tp = np.sum((y_true == 1) & (y_pred == 1))\n    fp = np.sum((y_true == 0) & (y_pred == 1))\n    fn = np.sum((y_true == 1) & (y_pred == 0))\n    denominator = (2 * tp) + fp + fn\n    return 0.0 if denominator == 0 else float((2 * tp) / denominator)\n\n\ndef evaluate_model(model, dataset):\n    keras_metrics = model.evaluate(dataset, verbose=0, return_dict=True)\n    raw_predictions = model.predict(dataset, verbose=0)\n\n    if isinstance(raw_predictions, dict):\n        predictions = raw_predictions\n    else:\n        predictions = dict(zip(model.output_names, raw_predictions))\n\n    true_gender, true_age = flatten_batches(dataset)\n    pred_gender = (predictions['gender_output'].reshape(-1) >= 0.5).astype(np.float32)\n    pred_age = predictions['age_output'].reshape(-1)\n\n    return {\n        **{k: float(v) for k, v in keras_metrics.items()},\n        'gender_accuracy': float(np.mean(true_gender == pred_gender)),\n        'gender_f1': binary_f1_score_np(true_gender, pred_gender),\n        'age_mae': float(np.mean(np.abs(true_age - pred_age))),\n    }\n

## 6. Training Loop\n\nMỗi backbone đều chạy 2 phase và lưu artifact riêng.\n

In [ ]:
def build_callbacks(run_dir: Path, phase_name: str, config: ExperimentConfig):\n    return [\n        tf.keras.callbacks.EarlyStopping(\n            monitor='val_loss',\n            patience=config.early_stopping_patience,\n            restore_best_weights=True,\n            verbose=1,\n        ),\n        tf.keras.callbacks.ReduceLROnPlateau(\n            monitor='val_loss',\n            factor=0.5,\n            patience=config.reduce_lr_patience,\n            min_lr=config.min_learning_rate,\n            verbose=1,\n        ),\n        tf.keras.callbacks.ModelCheckpoint(\n            filepath=str(run_dir / f'{phase_name}_best.keras'),\n            monitor='val_loss',\n            mode='min',\n            save_best_only=True,\n            verbose=1,\n        ),\n        tf.keras.callbacks.CSVLogger(str(run_dir / f'{phase_name}_history.csv')),\n    ]\n\n\ndef run_experiment(backbone_name: str, config: ExperimentConfig):\n    tf.keras.backend.clear_session()\n    tf.keras.utils.set_random_seed(config.seed)\n\n    run_dir = Path(config.output_dir) / backbone_name\n    run_dir.mkdir(parents=True, exist_ok=True)\n\n    split_summary = {\n        'train': describe_split(train_df),\n        'val': describe_split(val_df),\n        'test': describe_split(test_df),\n    }\n    (run_dir / 'split_summary.json').write_text(json.dumps(split_summary, indent=2))\n\n    model, backbone = build_multitask_model(backbone_name, image_size=config.image_size)\n\n    print(f'\\n===== {backbone_name.upper()} | PHASE 1: HEAD TRAINING =====')\n    compile_model(model, config.head_learning_rate, config.gender_loss_weight, config.age_loss_weight)\n    history1 = model.fit(\n        train_ds,\n        validation_data=val_ds,\n        epochs=config.phase1_epochs,\n        callbacks=build_callbacks(run_dir, 'phase1', config),\n        verbose=1,\n    )\n\n    print(f'\\n===== {backbone_name.upper()} | PHASE 2: FINE-TUNING =====')\n    configure_fine_tuning(backbone, config.fine_tune_layers, config.freeze_batch_norm)\n    compile_model(model, config.fine_tune_learning_rate, config.gender_loss_weight, config.age_loss_weight)\n    history2 = model.fit(\n        train_ds,\n        validation_data=val_ds,\n        epochs=config.phase2_epochs,\n        callbacks=build_callbacks(run_dir, 'phase2', config),\n        verbose=1,\n    )\n\n    model.save(run_dir / 'final_model.keras')\n\n    pd.DataFrame(history1.history).to_csv(run_dir / 'phase1_history_full.csv', index=False)\n    pd.DataFrame(history2.history).to_csv(run_dir / 'phase2_history_full.csv', index=False)\n\n    val_metrics = evaluate_model(model, val_ds)\n    test_metrics = evaluate_model(model, test_ds)\n\n    (run_dir / 'val_metrics.json').write_text(json.dumps(val_metrics, indent=2))\n    (run_dir / 'test_metrics.json').write_text(json.dumps(test_metrics, indent=2))\n\n    summary = {\n        'backbone': backbone_name,\n        'artifacts_dir': str(run_dir.resolve()),\n        'validation': val_metrics,\n        'test': test_metrics,\n    }\n    (run_dir / 'summary.json').write_text(json.dumps(summary, indent=2))\n\n    return summary\n

## 7. Run Training\n\nChạy cell này để train toàn bộ backbone. Nếu muốn test nhanh, bạn có thể rút `BACKBONES` còn 1 model trước.\n

In [ ]:
all_summaries = []\n\nfor backbone_name in BACKBONES:\n    summary = run_experiment(backbone_name, config)\n    all_summaries.append(summary)\n    print(json.dumps(summary, indent=2))\n\ncomparison_path = Path(config.output_dir) / 'comparison_summary.json'\ncomparison_path.write_text(json.dumps(all_summaries, indent=2))\nprint(f'Saved comparison summary to: {comparison_path}')\n

## 8. Compare Results\n

In [ ]:
comparison_df = pd.DataFrame([\n    {\n        'backbone': item['backbone'],\n        'val_gender_accuracy': item['validation']['gender_accuracy'],\n        'val_gender_f1': item['validation']['gender_f1'],\n        'val_age_mae': item['validation']['age_mae'],\n        'test_gender_accuracy': item['test']['gender_accuracy'],\n        'test_gender_f1': item['test']['gender_f1'],\n        'test_age_mae': item['test']['age_mae'],\n    }\n    for item in all_summaries\n])\n\ncomparison_df\n